# LCProp: portable reduced PR experiment (CPU)
Run in order in a fresh Python ≥3.10 kernel. See [README.md](README.md) for explicit setup.
This small example uses the authoritative headless engine; no notebook physics, GPU, Qt, widgets or automatic installation.
It is a startup demonstration, not a temporal/spatial convergence qualification.

In [ ]:
import sys
import importlib.util
import importlib.metadata
import platform
import hashlib
from pathlib import Path

def require_dependencies(find_spec=importlib.util.find_spec):
    if sys.version_info < (3, 10):
        raise RuntimeError("LCProp requires Python >=3.10; select a compatible kernel.")
    missing = [name for name in ("numpy", "scipy", "matplotlib", "lcprop")
               if find_spec(name) is None]
    if missing:
        raise RuntimeError("Missing notebook dependencies: " + ", ".join(missing)
                           + ". See notebooks/README.md; nothing was installed.")

require_dependencies()
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt
import lcprop

BASELINE_COMMIT = "985b80fbbc05616d4fe92f974ef0da24bf84e94c"
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "source checkout (distribution metadata unavailable)"

environment = {
    "python": sys.version, "executable": sys.executable, "platform": platform.platform(),
    "lcprop_version": package_version("lcprop"), "lcprop_source": str(lcprop.__file__),
    "notebook_baseline": BASELINE_COMMIT,
    "numpy": np.__version__, "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
}
print(environment)
print("CPU only. The baseline label is not proof of installed source identity; use the pinned setup.")

from copy import deepcopy
from dataclasses import dataclass

@dataclass(frozen=True)
class CompletedRun:
    request: object
    result: object
    run_data: object
    environment: dict
    elapsed_s: float

# Only successful execution replaces this record. Parameter/plot cells do not.
completed_run = None

def require_completed_run():
    snapshot = globals().get("completed_run")
    if snapshot is None:
        raise RuntimeError("Nothing to save or plot: execute a successful run first.")
    return snapshot

def plot_completed_run(snapshot):
    xy_field = snapshot.run_data.fields["output_intensity"]
    volume = snapshot.run_data.fields["optical_intensity_stack"]
    x = np.asarray(snapshot.run_data.geometry.x)
    y = np.asarray(snapshot.run_data.geometry.y)
    z = np.asarray(snapshot.run_data.geometry.z)
    iy = int(np.argmin(np.abs(y)))
    xy = xy_field.data
    xz = volume.data[:, :, iy]  # existing array view; no optical replay
    figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    im_xy = axes[0].pcolormesh(x, y, xy.T, shading="nearest")
    axes[0].set(xlabel=r"$x\ (\mu\mathrm{m})$", ylabel=r"$y\ (\mu\mathrm{m})$",
                title="Output optical intensity")
    axes[0].set_aspect("equal")
    im_xz = axes[1].pcolormesh(z, x, xz.T, shading="nearest")
    axes[1].set(xlabel=r"$z\ (\mu\mathrm{m})$", ylabel=r"$x\ (\mu\mathrm{m})$",
                title=f"Optical intensity, y={y[iy]:g} µm")
    for ax, im in zip(axes, (im_xy, im_xz)):
        figure.colorbar(im, ax=ax, label=r"$1/\mu\mathrm{m}^{2}$")
    figure.suptitle(f"Reduced nonlinear PR TD — accepted τ={snapshot.result.time_normalized:g}")
    return figure, axes, im_xy, im_xz


## Explicit scientific request
One 1 mW Gaussian beam, 0.633 µm wavelength, normal incidence, 12 µm beam-normal 1/e field radii
(1/e² intensity radii); 80 × 60 µm aperture, 64 × 48 grid, 20 µm propagation with 1 µm steps.
Reference index 2.4, gain-length product 0.1; no scattering. Physical dark-equivalent irradiance
0.01 W/cm², background 0 W/cm². Three steps of Δτ=0.001; τ is characteristic material time, **not seconds**.
Other material parameters use the displayed Product defaults.

In [ ]:
from dataclasses import asdict
from lcprop.core.backend import BackendSpec
from lcprop.core.context import GridSpec
from lcprop.core.beams import BeamChannel, BeamStack
from lcprop.pr.specs import PRMaterialSpec, PRSolverOptions, PRRunRequest
from lcprop.pr.gui.request_adapter import validate_pr_gui_request
from lcprop.persistence.experiments import (
    ExperimentCodecRegistry, write_experiment_file, read_experiment_file,
)
from lcprop.pr.experiment_codec import PR_TIMEDEPENDENT_EXPERIMENT_CODEC
from lcprop.pr.timedependent_transport_codec import PR_TIMEDEPENDENT_TRANSPORT_CODEC

request = PRRunRequest(
    grid=GridSpec(Nx=64, Ny=48, x_aperture_um=80., y_aperture_um=60.,
                  dz_um=1., z_length_um=20.),
    beams=BeamStack(channels=(BeamChannel(name="demo", power_mW=1., wavelength_um=0.633,
                                         w1_um=12., w2_um=12., coherence_group="demo-laser"),)),
    material=PRMaterialSpec.physical(dark_irradiance_W_cm2=0.01,
                                    uniform_irradiance_W_cm2=0.,
                                    gain_length_product=0.1),
    solver=PRSolverOptions(Nt=3, dt_normalized=0.001,
                           integrator="semi_implicit_trapezoidal"),
    backend=BackendSpec(backend="numpy", precision="float64", verbose=False),
)
# Existing preflight module is headless: no Qt application or LaunchPlane.
preflight = validate_pr_gui_request(request)
print("Preflight:", preflight)
print("Material (field names include canonical units):", asdict(request.material))
print("Solver:", asdict(request.solver))
print("Backend:", asdict(request.backend))


In [ ]:
from time import perf_counter
from lcprop.runners.local import LocalRunner
from lcprop.pr.operations import PR_TIMEDEPENDENT_OPERATION

progress_records = []
def report_progress(event):
    # Retain scalars only, never callback field arrays.
    record = {"status": event.status, "completed": event.completed_units,
              "total": event.total_units, "elapsed_s": event.elapsed_wall_time,
              "message": event.message}
    progress_records.append(record)
    print(f"{record['status']}: {record['completed']}/{record['total']}, "
          f"elapsed {record['elapsed_s']:.3f} s — {record['message']}")

executed_request = deepcopy(request)
started = perf_counter()
runner_result = LocalRunner().run_operation(
    PR_TIMEDEPENDENT_OPERATION, executed_request, progress_callback=report_progress,
)
result = runner_result.result
run_data = runner_result.run_data
elapsed_s = perf_counter() - started
print(f"Status={result.status}; accepted steps={result.completed_steps}; "
      f"material τ={result.time_normalized:g}; wall time={elapsed_s:.3f} s")
if result.status != "completed":
    raise RuntimeError("Example did not complete; inspect progress before treating it as a completed result.")
# Detach all saved data from mutable notebook presentation/execution variables.
# This bounded copy is appropriate only for the tiny Stage 1 demonstration.
completed_run = CompletedRun(
    request=deepcopy(executed_request), result=deepcopy(result),
    run_data=deepcopy(run_data), environment=deepcopy(environment), elapsed_s=elapsed_s,
)


## Retained optical products
These Product-generated final optical observations are not reconstructed material states.
Colorbars show **numerical optical intensity in 1/µm²**, not physical W/cm² or dimensionless total transport intensity.
The beam power and physical dark/background inputs above remain distinct.

In [ ]:
snapshot = require_completed_run()
figure, axes, im_xy, im_xz = plot_completed_run(snapshot)
plt.show()


## Save explicitly
Call `save_demo(Path("lcprop-reduced-pr-demo"))` with a writable **new** directory.
This saves a strict .lcprop.json experiment, existing checksummed LCProp request/result package,
selected xy/xz NPZ with coordinates, figure and environment/hash manifest. Existing folders are never overwritten.
The full package is practical only for this tiny example, not a large-grid retention policy.
No Drive mounting or upload occurs. Colab storage is temporary; download/copy outputs before ending the session.
Exports use the most recent successful completed-run snapshot, even if parameters have since changed.
Execute again to save new parameters. Plotting and saving regenerate figures from that same snapshot;
failed executions do not replace it. Saving before a successful execution is rejected.


In [ ]:
import json
from lcprop.transport.codecs import TransportCodecRegistry
from lcprop.transport.io import write_request_package, read_request_package, write_result_package

def save_demo(directory):
    snapshot = require_completed_run()
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=False)
    experiments = ExperimentCodecRegistry()
    experiments.register(PR_TIMEDEPENDENT_EXPERIMENT_CODEC)
    write_experiment_file(directory / "experiment.lcprop.json", snapshot.request,
                          material_id="pr", workflow_id="pr_timedependent", registry=experiments)
    codecs = TransportCodecRegistry()
    codecs.register(PR_TIMEDEPENDENT_TRANSPORT_CODEC)
    write_request_package(directory, registry=codecs, material_id="pr",
                          workflow_id="pr_timedependent", request=snapshot.request,
                          run_id="portable-reduced-pr-demo", execution_target="local",
                          result_policy="full", provenance={"notebook_environment": snapshot.environment})
    decoded = read_request_package(directory, registry=codecs)
    write_result_package(directory, codec=decoded.codec, result=snapshot.result,
                         request_envelope=decoded.envelope)
    data = snapshot.run_data
    x, y, z = (np.asarray(getattr(data.geometry, axis)) for axis in ("x", "y", "z"))
    iy = int(np.argmin(np.abs(y)))
    xy = data.fields["output_intensity"].data
    xz = data.fields["optical_intensity_stack"].data[:, :, iy]
    np.savez(directory / "selected_intensity.npz", output_xy=xy, optical_xz=xz,
             x_um=x, y_um=y, z_um=z, y_cut_um=y[iy])
    export_figure, *_ = plot_completed_run(snapshot)
    try:
        export_figure.savefig(directory / "intensity.png", dpi=140)
    finally:
        plt.close(export_figure)
    hashes = {str(p.relative_to(directory)): hashlib.sha256(p.read_bytes()).hexdigest()
              for p in sorted(directory.rglob("*")) if p.is_file()}
    (directory / "notebook-manifest.json").write_text(json.dumps(
        {"environment": snapshot.environment, "status": snapshot.result.status, "elapsed_s": snapshot.elapsed_s,
         "accepted_steps": snapshot.result.completed_steps, "material_time": snapshot.result.time_normalized,
         "sha256": hashes}, indent=2, allow_nan=False) + "\n")
    return directory
